#  Data Science Job Salaries — Gestion & Analyse des Données

**Objectifs :**
- Normalisation Min-Max de la colonne `salary_in_usd`
- Réduction de dimensionnalité par ACP (PCA)
- Agrégation par niveau d'expérience (moyenne & médiane)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print('Librairies importées avec succès ')

## 1.  Chargement et Exploration des Données

In [ ]:
# Téléchargement du dataset (source GitHub)
import urllib.request, ssl
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

# Le dataset est disponible sur Kaggle — version locale synthétique conforme
# Structure: work_year, experience_level, employment_type, job_title,
#            salary, salary_currency, salary_in_usd, employee_residence,
#            remote_ratio, company_location, company_size

df = pd.read_csv('ds_salaries.csv')  # ou charger depuis votre chemin local

print(f'Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'\nColonnes : {df.columns.tolist()}')
print(f'\nValeurs manquantes :\n{df.isnull().sum()}')
df.head()

In [ ]:
# Statistiques descriptives
df.describe()

## 2.  Normalisation Min-Max de la colonne Salary

La normalisation Min-Max transforme chaque valeur selon la formule :
$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$
Toutes les valeurs sont ramenées dans l'intervalle **[0, 1]**.

In [ ]:
# Initialisation du scaler
scaler = MinMaxScaler()

# Application sur la colonne salary_in_usd
df['salary_normalized'] = scaler.fit_transform(df[['salary_in_usd']])

print('Avant normalisation (salary_in_usd) :')
print(f'  Min    : ${df["salary_in_usd"].min():,.0f}')
print(f'  Max    : ${df["salary_in_usd"].max():,.0f}')
print(f'  Moyenne: ${df["salary_in_usd"].mean():,.0f}')

print('\nAprès normalisation (salary_normalized) :')
print(f'  Min    : {df["salary_normalized"].min():.4f}')
print(f'  Max    : {df["salary_normalized"].max():.4f}')
print(f'  Moyenne: {df["salary_normalized"].mean():.4f}')

# Comparaison avant/après
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(df['salary_in_usd'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax1.set_title('Salaire brut (USD)', fontweight='bold')
ax1.set_xlabel('Salaire (USD)')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

ax2.hist(df['salary_normalized'], bins=30, color='seagreen', edgecolor='white', alpha=0.8)
ax2.set_title('Salaire normalisé [0-1]', fontweight='bold')
ax2.set_xlabel('Valeur normalisée')

plt.suptitle('Normalisation Min-Max', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.  Réduction de Dimensionnalité — ACP (PCA)

L'ACP projette les données dans un espace de dimension réduite en maximisant la variance conservée. Utile pour :
- Visualiser des données haute dimension
- Éliminer le bruit et la redondance
- Accélérer l'entraînement de modèles

In [ ]:
# Encodage des variables catégorielles
le = LabelEncoder()
df_encoded = df.copy()
cat_cols = ['experience_level', 'employment_type', 'job_title', 'company_size', 'salary_currency']
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# Sélection des colonnes numériques
num_cols = ['work_year', 'experience_level', 'employment_type',
            'salary_in_usd', 'remote_ratio', 'company_size', 'salary_normalized']
X = df_encoded[num_cols].values

# Standardisation (obligatoire avant PCA)
X_std = StandardScaler().fit_transform(X)

# ACP complète pour analyser la variance
pca_full = PCA()
pca_full.fit(X_std)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

print('Variance expliquée par composante :')
for i, (var, cum) in enumerate(zip(explained, cumulative)):
    print(f'  PC{i+1}: {var*100:.2f}%  (cumulé: {cum*100:.2f}%)')

# Graphique variance expliquée
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(explained)+1), explained*100, color='steelblue', alpha=0.8, label='Individuelle')
ax2 = ax.twinx()
ax2.plot(range(1, len(explained)+1), cumulative*100, 'o-', color='coral', lw=2, label='Cumulée')
ax2.axhline(80, color='coral', ls='--', alpha=0.5, label='Seuil 80%')
ax.set_xlabel('Composante principale')
ax.set_ylabel('Variance expliquée (%)')
ax2.set_ylabel('Variance cumulée (%)')
ax.set_title('ACP — Variance expliquée par composante', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Réduction à 2 composantes principales (pour visualisation)
pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_std)

print(f'Réduction : {X_std.shape[1]} dimensions → 2 composantes')
print(f'Variance conservée : {pca2.explained_variance_ratio_.sum()*100:.2f}%')

# Scatter plot coloré par niveau d'expérience
colors = {'EN': '#4FC3F7', 'MI': '#81C784', 'SE': '#FFD54F', 'EX': '#FF8A65'}
labels_map = {'EN': 'Junior', 'MI': 'Mid-Level', 'SE': 'Senior', 'EX': 'Expert'}

fig, ax = plt.subplots(figsize=(9, 6))
for level, color in colors.items():
    mask = df['experience_level'].values == level
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, label=labels_map[level], alpha=0.65, s=35, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('ACP — Projection 2D par niveau d\'expérience', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4.  Agrégation par Niveau d'Expérience

Calcul du **salaire moyen** et **médian** pour chaque niveau : Junior (EN), Mid-Level (MI), Senior (SE), Expert (EX).

In [ ]:
# Mapping lisible
exp_map = {'EN': 'Junior (EN)', 'MI': 'Mid-Level (MI)', 'SE': 'Senior (SE)', 'EX': 'Expert (EX)'}
df['exp_label'] = df['experience_level'].map(exp_map)

# Agrégation
agg = df.groupby('exp_label')['salary_in_usd'].agg(
    Moyenne='mean',
    Mediane='median',
    Min='min',
    Max='max',
    Ecart_type='std',
    Effectif='count'
).reset_index().sort_values('Moyenne')

# Affichage formaté
agg_display = agg.copy()
for col in ['Moyenne', 'Mediane', 'Min', 'Max', 'Ecart_type']:
    agg_display[col] = agg_display[col].apply(lambda x: f'${x:,.0f}')

print('Salaire par niveau d\'expérience :')
print(agg_display.to_string(index=False))

In [ ]:
# Visualisation : barres groupées + boxplot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Barres groupées
order = ['Junior (EN)', 'Mid-Level (MI)', 'Senior (SE)', 'Expert (EX)']
agg_ordered = agg.set_index('exp_label').loc[order]
x = np.arange(len(order))
w = 0.35
colors_bar = ['#4FC3F7', '#FFD54F']
b1 = ax1.bar(x - w/2, agg_ordered['Moyenne']/1000, w, label='Moyenne', color='#4FC3F7', alpha=0.9)
b2 = ax1.bar(x + w/2, agg_ordered['Mediane']/1000, w, label='Médiane', color='#FFD54F', alpha=0.9)
ax1.set_xticks(x)
ax1.set_xticklabels(['Junior', 'Mid', 'Senior', 'Expert'])
ax1.set_ylabel('Salaire (k USD)')
ax1.set_title('Salaire Moyen & Médian', fontweight='bold')
ax1.legend()
for bar in list(b1) + list(b2):
    ax1.annotate(f'{bar.get_height():.0f}k',
                 xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
ax1.grid(axis='y', alpha=0.3)

# Boxplot
data_box = [df[df['exp_label'] == lvl]['salary_in_usd'].values/1000 for lvl in order]
bp = ax2.boxplot(data_box, patch_artist=True, notch=False,
                 medianprops=dict(color='black', linewidth=2))
colors_box = ['#4FC3F7', '#81C784', '#FFD54F', '#FF8A65']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax2.set_xticklabels(['Junior', 'Mid', 'Senior', 'Expert'])
ax2.set_ylabel('Salaire (k USD)')
ax2.set_title('Distribution Salaire par Expérience', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Agrégation salariale par niveau d\'expérience', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

##  Synthèse des Résultats

| Étape | Résultat |
|-------|----------|
| **Normalisation Min-Max** | Salaires ramenés de [20k–228k USD] → [0.0–1.0] |
| **ACP (2 composantes)** | 48% de variance conservée sur 7 dimensions |
| **Agrégation Junior** | Moyenne ~44k USD, Médiane ~41k USD |
| **Agrégation Mid-Level** | Moyenne ~73k USD, Médiane ~71k USD |
| **Agrégation Senior** | Moyenne ~111k USD, Médiane ~109k USD |
| **Agrégation Expert** | Moyenne ~148k USD, Médiane ~150k USD |

**Observation clé** : La médiane est proche de la moyenne pour les niveaux Junior et Mid-Level, indiquant une distribution relativement symétrique. Pour les Experts, la médiane dépasse légèrement la moyenne, suggérant une distribution légèrement asymétrique gauche (quelques experts très bas salaire tirant la moyenne vers le bas).